In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('../data/processed/fraud_data_enriched.csv')
df['signup_time'] = pd.to_datetime(df['signup_time'])
df['purchase_time'] = pd.to_datetime(df['purchase_time'])

# Time duration from registration to checkout (in seconds)
df['time_since_signup'] = (df['purchase_time'] - df['signup_time']).dt.total_seconds()

# Extract seasonal/cyclical time blocks
df['hour_of_day'] = df['purchase_time'].dt.hour
df['day_of_week'] = df['purchase_time'].dt.dayofweek

In [3]:
# Calculate transaction velocity per device ID across the entire logging period
df['device_velocity'] = df.groupby('device_id')['user_id'].transform('count')

# Calculate transaction velocity per IP address
df['ip_velocity'] = df.groupby('ip_address')['user_id'].transform('count')

In [4]:
# One-hot encode high-value categories, bundle tail variables into 'Other' if necessary
categorical_cols = ['source', 'browser', 'sex']
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# Select engineered and original continuous variables for scaling
numerical_features = ['purchase_value', 'age', 'time_since_signup', 'hour_of_day', 'day_of_week', 'device_velocity', 'ip_velocity']

scaler = StandardScaler()
df_encoded[numerical_features] = scaler.fit_transform(df_encoded[numerical_features])

# Drop features that can cause structural leaks or have been parsed out
features_to_drop = ['user_id', 'signup_time', 'purchase_time', 'device_id', 'ip_address', 'country', 'ip_int']
final_fraud_features = df_encoded.drop(columns=features_to_drop)

# Export finalized dataset for training
final_fraud_features.to_csv('../data/processed/fraud_features_final.csv', index=False)
print("E-commerce engineering complete. Final shape:", final_fraud_features.shape)

E-commerce engineering complete. Final shape: (151112, 15)
